### 1.生成训练数据

In [5]:
import torch

# 设定设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 随机生成100个含有3个特征的输入
inputs = torch.rand(100, 3)
# 设定权重和偏置
weights = torch.tensor([[1.1], [2.2], [3.3]])
bias = torch.tensor([4.4])

# 对应的标签值为这3个特征的线性组合加上一个偏置项再加上一些噪声
targets = inputs @ weights + bias + 0.1*torch.rand(100, 1)

### 2. 初始化线性回归参数

In [6]:
w = torch.rand((3, 1), requires_grad=True, device=device)
b = torch.rand((1,), requires_grad=True, device=device)

### 3. 进行训练

In [12]:
# 将数据移至相同设备
inputs = inputs.to(device)
targets = targets.to(device)

# 设置超参数
epoch_num = 10000
lr = 0.003

# 定义损失函数
def Loss(outputs, targets):
    differ = outputs-targets
    square_num = torch.square(differ)
    mean_loss = torch.mean(square_num)

    return mean_loss

for epoch in range(epoch_num):
    

    y_pred = inputs @ w + b
    loss = Loss(y_pred,targets)
    
    # 反向传播
    loss.backward()

    # 更新参数
    with torch.no_grad():
        w -= lr*w.grad
        b -= lr*b.grad

        # 清空上一轮梯度
        w.grad.zero_()
        b.grad.zero_()
    if epoch % 100 == 0:
        print('epoch',f'{epoch}:','loss',f'{loss.item():.5f}')

print(f'训练后权重:')
print("w:", w.detach())
print("b:", b.detach())



epoch 0: loss 0.04818
epoch 100: loss 0.04348
epoch 200: loss 0.03928
epoch 300: loss 0.03551
epoch 400: loss 0.03212
epoch 500: loss 0.02908
epoch 600: loss 0.02635
epoch 700: loss 0.02390
epoch 800: loss 0.02170
epoch 900: loss 0.01972
epoch 1000: loss 0.01794
epoch 1100: loss 0.01634
epoch 1200: loss 0.01490
epoch 1300: loss 0.01360
epoch 1400: loss 0.01243
epoch 1500: loss 0.01138
epoch 1600: loss 0.01042
epoch 1700: loss 0.00957
epoch 1800: loss 0.00879
epoch 1900: loss 0.00809
epoch 2000: loss 0.00746
epoch 2100: loss 0.00689
epoch 2200: loss 0.00637
epoch 2300: loss 0.00590
epoch 2400: loss 0.00548
epoch 2500: loss 0.00509
epoch 2600: loss 0.00474
epoch 2700: loss 0.00443
epoch 2800: loss 0.00414
epoch 2900: loss 0.00388
epoch 3000: loss 0.00364
epoch 3100: loss 0.00342
epoch 3200: loss 0.00323
epoch 3300: loss 0.00305
epoch 3400: loss 0.00288
epoch 3500: loss 0.00273
epoch 3600: loss 0.00259
epoch 3700: loss 0.00247
epoch 3800: loss 0.00236
epoch 3900: loss 0.00225
epoch 4000: 

- loss.backward() 只做一件事：沿计算图反向传播，把梯度算出来并累加到各个参数的 .grad 里（比如 w.grad、b.grad），不负责更新参数值。

- with 是 Python 的“上下文管理器”语法：进入一段代码前自动做准备工作，离开这段代码时（不管正常结束还是报错）自动做清理/收尾。

- detach() 的作用：把一个张量从计算图里“摘出来”，返回一个新的张量视图，它和原张量共享数据，但 不再参与梯度计算（requires_grad=False）。
    - 所以不要用 w = w.detach() 去替换训练用的参数。

    你可以把它理解为：
    “我只想把这个数/这个张量拿去打印、保存、转成 numpy，不要让 autograd 继续追踪它。”